# 02 — Factor Diagnostics

Summary stats, correlations, and sanity checks for SMB, HML, WML, and Amihud ILLIQ.

Signals use information through $t-1$ only; characteristics are winsorized at 1%/99% before z-scores.

In [ ]:
from pathlib import Path
import polars as pl

from src.data.store import read_partitioned_parquet
from src.data.paths import PROCESSED_RETURNS_DIR, PROCESSED_FACTORS_DIR
from src.factors.combine import build_factors

returns = read_partitioned_parquet(PROCESSED_RETURNS_DIR)
print("returns rows:", returns.height)
returns.head()

In [ ]:
if returns.height:
    result = build_factors(returns, amihud_window=30, winsorize=True, persist=True)
    chars, fac = result.characteristics, result.factor_returns
else:
    print("No processed returns — run Phase-2 pipeline first, or build from fixtures.")
    chars = fac = None

## Factor return summary

In [ ]:
if fac is not None and fac.height:
    cols = [c for c in ("SMB", "HML", "WML", "ILLIQ") if c in fac.columns]
    display(fac.select(cols).describe())
    display(fac.select(cols).corr())
    fac.head()

## Amihud signal coverage

In [ ]:
if chars is not None and "illiq_signal" in chars.columns:
    cov = (
        chars.group_by("trade_date")
        .agg(
            pl.col("illiq_signal").is_not_null().mean().alias("pct_non_null"),
            pl.col("illiq_signal").mean().alias("mean_signal"),
        )
        .sort("trade_date")
    )
    display(cov.describe())
    cov.tail(10)